In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [ ]:
import sys
import subprocess

# 1. Install without crashing the session
# We use --no-cache-dir to ensure we don't hit disk limits
print("Starting installation...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "uv", "-q"])
subprocess.check_call(["uv", "pip", "install", "--system", "autogluon.tabular[all]", "datasets>=2.16.0", "pyarrow<20", "-q"])

# 2. THE SECRET SAUCE: Refresh the site packages in the current process
import site
from importlib import reload
reload(site)

# 3. Test the import immediately
try:
    from autogluon.tabular import TabularPredictor
    print("AutoGluon successfully loaded without a restart!")
except Exception as e:
    print(f"Import still failing: {e}")

In [ ]:
# pip install autogluon -q

In [5]:
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor

# 1. Load Data
# Ensure these paths match your Kaggle input folder structure
train_data = TabularDataset('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test_data = TabularDataset('/kaggle/input/competitions/playground-series-s6e4/test.csv')

# 2. Configuration
label = 'Irrigation_Need' # <--- Update this
id_column = 'id'             # <--- Update this (e.g., 'id', 'PassengerId')
eval_metric = 'balanced_accuracy'

# 3. Initialize and Train
# verbosity=2 prints the training stages and progress of each model
predictor = TabularPredictor(
    label=label, 
    eval_metric=eval_metric,
    verbosity=2 
).fit(
    train_data,
    presets='best_quality', # Enables stacking and bagging for high LB rank
    time_limit=3600 * 0.5,    # 3 hours (adjust based on your Kaggle quota)
    num_stack_levels=1,     # Force at least 1 level of stacking for accuracy
    dynamic_stacking=True   # Allows AG to decide if more stacking is beneficial
)

# 4. View Training Progress Leaderboard
# This shows you how each individual model performed during the 'stages'
print("\n--- Internal Leaderboard (Validation Scores) ---")
lb = predictor.leaderboard(silent=False)
display(lb)

# 5. Generate Predictions
# Use 'predict_proba' if the competition requires probabilities, 
# but usually 'predict' is used for class labels.
print("\nGenerating predictions for test set...")
predictions = predictor.predict(test_data)

# 6. Final Submission
submission = pd.DataFrame({
    id_column: test_data[id_column],
    label: predictions
})

submission.to_csv('submission.csv', index=False)
print("Success: submission.csv generated.")

Loaded data from: /kaggle/input/competitions/playground-series-s6e4/train.csv | Columns = 21 / 21 | Rows = 630000 -> 630000
Loaded data from: /kaggle/input/competitions/playground-series-s6e4/test.csv | Columns = 20 / 20 | Rows = 270000 -> 270000
No path specified. Models will be saved in: "AutogluonModels/ag-20260425_041046"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sat Jan 17 11:20:45 UTC 2026
CPU Count:          4
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       24.37 GB / 31.35 GB (77.7%)
Disk Space Avail:   19.38 GB / 19.52 GB (99.3%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoG


--- Internal Leaderboard (Validation Scores) ---
                    model  score_val        eval_metric  pred_time_val    fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  NeuralNetFastAI_BAG_L1   0.957823  balanced_accuracy      12.019862  468.500768               12.019862         468.500768            1       True          1
1     WeightedEnsemble_L2   0.957823  balanced_accuracy      12.167687  482.319925                0.147825          13.819157            2       True          4
2       LightGBMXT_BAG_L1   0.935280  balanced_accuracy      44.650894  167.921545               44.650894         167.921545            1       True          2
3         LightGBM_BAG_L1   0.333333  balanced_accuracy       0.581166   18.093281                0.581166          18.093281            1       True          3


,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,NeuralNetFastAI_BAG_L1,0.957823,balanced_accuracy,12.019862,468.500768,12.019862,468.500768,1,True,1
1,WeightedEnsemble_L2,0.957823,balanced_accuracy,12.167687,482.319925,0.147825,13.819157,2,True,4
2,LightGBMXT_BAG_L1,0.935280,balanced_accuracy,44.650894,167.921545,44.650894,167.921545,1,True,2
3,LightGBM_BAG_L1,0.333333,balanced_accuracy,0.581166,18.093281,0.581166,18.093281,1,True,3



Generating predictions for test set...
Success: submission.csv generated.
